# 04 — Data Quality Checks
Independent verification only. This notebook reads outputs and does not modify pipeline data.


In [0]:
from pyspark.sql import functions as F
BASE_PATH = "abfss://edtech@edtechpipline26.dfs.core.windows.net"
validated = spark.read.format("delta").load(f"{BASE_PATH}/interim/validated")
rejected = spark.read.format("delta").load(f"{BASE_PATH}/interim/rejected")
final = spark.read.format("delta").load(f"{BASE_PATH}/processed/final")


## Record counts


In [0]:
print("Validated:", validated.count())
print("Rejected:", rejected.count())
print("Final:", final.count())


## Category and source distributions


In [0]:
display(final.groupBy("category").count().orderBy(F.desc("count")))
display(final.groupBy("source").count().orderBy(F.desc("count")))


## Critical quality checks


In [0]:
unexpected_categories = final.filter(~F.col("category").isin("AI","Data","Cloud")).count()
duplicate_urls = final.groupBy("url").count().filter(F.col("count") > 1).count()
missing_critical = final.filter(
    F.col("title").isNull() | (F.trim(F.col("title")) == "") |
    F.col("url").isNull() | (F.trim(F.col("url")) == "") |
    F.col("content").isNull() | (F.trim(F.col("content")) == "")
).count()
print("Unexpected category rows:", unexpected_categories)
print("Duplicate URLs:", duplicate_urls)
print("Missing critical fields:", missing_critical)


## Word counts and long-form distribution


In [0]:
display(final.groupBy("source").agg(
    F.round(F.avg("word_count"),1).alias("avg_words"),
    F.min("word_count").alias("min_words"),
    F.max("word_count").alias("max_words")
))
display(final.groupBy("is_long_form").count())


## Publication dates and rejected records


In [0]:
display(final.groupBy("source").agg(
    F.sum(F.when(F.col("publication_date").isNull(),1).otherwise(0)).alias("missing_dates"),
    F.min("publication_date").alias("earliest_date"),
    F.max("publication_date").alias("latest_date")
))
display(rejected.select("source","title","url","rejection_reason"))


## Final pass/fail summary


In [0]:
expected_sources = {"dev.to","freeCodeCamp","GeeksforGeeks","Medium","Pluralsight"}
actual_sources = {r["source"] for r in final.select("source").distinct().collect()}
checks = [
    ("Gold row count equals validated row count", final.count() == validated.count()),
    ("Only AI/Data/Cloud categories", unexpected_categories == 0),
    ("All five sources present", actual_sources == expected_sources),
    ("No duplicate URLs", duplicate_urls == 0),
    ("No missing critical fields", missing_critical == 0),
]
for name, passed in checks:
    print(("PASS" if passed else "FAIL"), "-", name)
